# TradeSense · TGNN branch · Trade-price diagnostic (read-only) → `08_tgnn_tardis_trade_diagnostic_v1`

**Purpose.** The first Tardis audit (`07_…`) reported that about 47.6 % of sampled BTCUSDT trades printed more than one tick away from the rebuilt best bid/ask, and its "near best bid/ask" classes **overlapped** (their shares added up to about 125 %). This notebook re-examines trade prices against the rebuilt book **with mutually exclusive categories** and investigates *why* so many trades are away from the touch — **without assuming the answer**.

**What it does NOT do.** It downloads nothing (it never contacts the internet), reads only the files already saved by `07_…`, never modifies them, trains nothing, builds no graphs and creates no synthetic events. Its own results go to a new folder `TGNN_Tardis_TradeDiag_BTCUSDT_2024-05-01_v1`; it refuses to run if that folder already exists. The earlier audit's thresholds and results are not touched or re-judged.

**Reference book.** The **full-depth (unpruned)** replay is the reference (it was the variant consistent with Tardis' own `book_snapshot_25` in the first audit; the 1000-level-pruned variant was not). Here it is replayed once more, with trades interleaved, and re-checked against the earlier audit's recorded numbers and against `book_snapshot_25` best bid/ask.

**Categories (mutually exclusive, defined relative to the side the taker hits).** A *sell-taker* hits the bid, a *buy-taker* lifts the ask, so for each trade the "touch" is the best bid (sell) or the best ask (buy). With `d` = how many ticks the trade price is *deeper* than that touch:
`through_opposite` (price at/beyond the other side's touch) · `inside_spread` (strictly between, better than the touch) · `at_touch` (d = 0) · `deeper_1` (d = 1) · `deeper_2_5` · `deeper_6_20` · `deeper_21_100` · `deeper_gt100`. The old problem — with a one-tick spread the ask is "within one tick" of the bid — cannot occur because every trade is classified against **one** touch only.

**The book state used.** Primary: the book after the last *completed* message whose `local_timestamp` is **strictly before** the trade's `local_timestamp` (so a message is never peeked at before the trade). Alternatives (exchange timestamps, time shifts) are examined separately in section 6.

**Candidate explanations, each examined on its own (none is assumed):**
* **H-timing** — clocks/latency: the trade and depth feeds are separate connections, so the book state "at" a trade may be misaligned. Tested with local-clock and exchange-clock alignment, time shifts of ±2 s, and an "oracle" that looks at the best-matching book state within ±1 s.
* **H-sweep** — one taker order filling several price levels, printed as several trades. Tested with a **heuristic** grouping (same side, consecutive trade IDs, same exchange timestamp, price never improving). **This grouping is NOT verified order identity** — Tardis' CSV has no order IDs. Also tested mechanically: was the first level's displayed quantity actually exhausted before the price moved deeper?
* **H-stale** — the depth feed is batched (about 10 messages per second), so a book state can be up to ~100 ms old. Tested by trade-age bins and by position within a book-update interval.
* **H-tolerance** — "1 tick" is $0.01 on a ~$57,000 asset (≈ 0.0018 bps), an extremely tight yardstick. Tested by distance in ticks and in bps.
* **H-side** — the `side` column might not be the taker side. Tested against the book (original vs swapped labels; which side's touch quantity shrinks after a trade).
* **H-reconstruction** — the rebuilt book might simply be wrong. Tested with the best bid/ask re-check against `book_snapshot_25` and with the share of trades priced on the wrong side of the book.

**Evidence grading rules, frozen before the data is seen** (descriptive grades, not p-values): let *Q* = trades at least 2 ticks deeper than the touch under the primary alignment.
* H-timing: share of Q explained by the oracle (within 1 tick of the aggressed touch at some state within ±1 s, local or exchange clock) — **SUPPORTED ≥ 50 %, PARTIAL 20–50 %, NOT SUPPORTED < 20 %**.
* H-sweep: share of Q that are non-first fills of a heuristic group with ≥ 2 distinct prices — same cut-offs. (A mechanics-consistency rate is reported alongside.)
* H-stale: P(Q | age ≥ 50 ms) ÷ P(Q | age < 10 ms) — **SUPPORTED ≥ 1.5, NOT SUPPORTED ≤ 1.1, otherwise PARTIAL**.
* H-tolerance: share of Q within 1 bp of the touch — **"1 tick is too tight a yardstick" if ≥ 90 %**.
* H-side: consistent share (at-touch or deeper) with the original labels minus with swapped labels — **SUPPORTED (labels are the taker side) if ≥ 0.20**.
* H-reconstruction: **SUPPORTED (book is the problem) if** the best bid/ask re-check fails **or** more than 1 % of trades are priced through the opposite touch.
* The remainder of Q not explained by timing **or** sweep is reported as **residual**.

**Recommendation rule (frozen).** *PROCEED to ask for approval of the remaining free days* only if every E and D test passes, the best-bid/ask re-check passes, H-side is supported, and ≤ 1 % of trades are priced through the opposite touch. If the residual exceeds 20 % of Q, the recommendation carries the caveat that trades must not be used for exact price-level alignment with the book. Otherwise: **HOLD**. The recommendation is advice; nothing is downloaded without your approval.

**Test groups.** **E** = notebook execution tests, **D** = input-data integrity tests (re-check of the saved files). There is no reconstruction verdict here: passing E and D does not validate anything about trades or the book.

## 1 · Setup, read-only guard, and a fingerprint of everything that already exists

In [ ]:
import os, sys, io, re, gc, json, time, math, gzip, uuid, shutil, hashlib, heapq, warnings, resource
from array import array
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
# NOTE: this notebook deliberately imports no HTTP library and never touches the network.
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
assert BASE.is_dir(), f'Drive folder not found: {BASE}'
SRC = BASE / 'TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1'
OUT_FINAL = BASE / 'TGNN_Tardis_TradeDiag_BTCUSDT_2024-05-01_v1'
STAGE = BASE / ('.tardis_diag_stage_' + uuid.uuid4().hex[:8])
assert SRC.is_dir(), f'Source folder from notebook 07 not found: {SRC}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
EXCHANGE, SYMBOL, DAY = 'binance', 'BTCUSDT', '2024-05-01'
DAY0_US = int(datetime(2024, 5, 1, tzinfo=timezone.utc).timestamp() * 1_000_000); DAY1_US = DAY0_US + 86_400_000_000
FILES = {dt: SRC / 'raw' / f'{EXCHANGE}_{dt}_{DAY}_{SYMBOL}.csv.gz' for dt in ('incremental_book_L2', 'trades', 'book_snapshot_25')}
for k in ('incremental_book_L2', 'trades'): assert FILES[k].is_file(), f'missing {FILES[k]}'
HAVE_S25 = FILES['book_snapshot_25'].is_file()
UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); STAGES = {}; REPORT = {}
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def peak_mem_gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
def tick_(name, t0): STAGES[name] = {'seconds': round(time.time() - t0, 1), 'peak_mem_gb_so_far': round(peak_mem_gb(), 2)}; print(f'[{name}] {STAGES[name]["seconds"]} s | peak memory so far {STAGES[name]["peak_mem_gb_so_far"]} GB')
def sh(a, b): return (a / b) if b else float('nan')
t0 = time.time(); PIN0 = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file()}
print(f'fingerprinted {len(PIN0)} existing files (including the 07 output folder); they will be re-checked at the end'); tick_('fingerprint', t0)
MAN = json.loads((SRC / 'manifest.json').read_text()); SRC_REPORT = json.loads((SRC / 'reports' / 'audit_report.json').read_text()); SRC_SHARE = (SRC / 'reports' / 'share_block.txt').read_text()
print('source folder:', SRC); print('book_snapshot_25 present:', HAVE_S25)

Mounted at /content/drive
fingerprinted 759 existing files (including the 07 output folder); they will be re-checked at the end
[fingerprint] 27.6 s | peak memory so far 0.17 GB
source folder: /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1
book_snapshot_25 present: True


## 2 · Load the saved files (read-only) and check them against the SHA-256 recorded by notebook 07

In [ ]:
t0 = time.time()
SHA_CHECK = {}
for dt, p in FILES.items():
    if not p.is_file(): continue
    key = 'raw/' + p.name; SHA_CHECK[dt] = {'file': p.name, 'recorded': MAN.get(key), 'now': sha256_file(p)}; SHA_CHECK[dt]['equal'] = SHA_CHECK[dt]['recorded'] == SHA_CHECK[dt]['now']
    print(dt, 'sha256 equal to manifest:', SHA_CHECK[dt]['equal'])
tick_('sha256_recheck', t0)
t0 = time.time()
try: tr_tbl = pacsv.read_csv(FILES['trades'], convert_options=pacsv.ConvertOptions(column_types={'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64(), 'id': pa.int64(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()}))
except Exception as e: raise RuntimeError('trade ids are not plain integers -> STOP and paste this to Claude: ' + str(e)[:200])
tE = tr_tbl.column('timestamp').to_numpy(); tL = tr_tbl.column('local_timestamp').to_numpy(); tid = tr_tbl.column('id').to_numpy(); tpx = tr_tbl.column('price').to_numpy(); tam = tr_tbl.column('amount').to_numpy()
side_s = tr_tbl.column('side'); tside = np.full(len(tE), 2, np.int8); tside[pc.equal(side_s, 'buy').to_numpy(zero_copy_only=False).astype(bool)] = 0; tside[pc.equal(side_s, 'sell').to_numpy(zero_copy_only=False).astype(bool)] = 1
NT = len(tE); del tr_tbl; gc.collect(); tick_('load_trades', t0); print(f'trades loaded: {NT:,} | buy {int((tside==0).sum()):,} sell {int((tside==1).sum()):,} other {int((tside==2).sum()):,}')
TRADES_SORTED_LOCAL = bool(np.all(np.diff(tL) >= 0)); TRADES_SORTED_EXCH = bool(np.all(np.diff(tE) >= 0)); print('trades sorted by local ts:', TRADES_SORTED_LOCAL, '| by exchange ts:', TRADES_SORTED_EXCH)
if not TRADES_SORTED_LOCAL: raise RuntimeError('trades are not in local-timestamp order: STOP and paste this to Claude.')
BOOK_TYPES = {'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64(), 'is_snapshot': pa.bool_(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()}
def book_reader(block_mb=48): return pacsv.open_csv(FILES['incremental_book_L2'], read_options=pacsv.ReadOptions(block_size=block_mb << 20), convert_options=pacsv.ConvertOptions(column_types=BOOK_TYPES, strings_can_be_null=False, true_values=['true', 'True', 'TRUE'], false_values=['false', 'False', 'FALSE']))
# tick size from the data itself: the LARGEST "nice" step (1, 2, 5 x 10^k) on which every price in the first book block and in the trades lies
first_blk = next(iter(book_reader(16))); pr0 = np.unique(first_blk.column('price').to_numpy())
def off_grid_share(p, tk): x = p / tk; return float(np.mean(np.abs(x - np.rint(x)) > 1e-4))
CANDS = sorted({m * 10.0 ** k for k in range(-8, 3) for m in (1, 2, 5)}, reverse=True); probe_p = np.concatenate([pr0, tpx[:200_000]])
TICK = next((c for c in CANDS if off_grid_share(probe_p, c) == 0.0), None)
if TICK is None: raise RuntimeError('no tick size fits all prices: STOP and paste this to Claude.')
TICK = float(TICK); print('tick size inferred from the data:', TICK)
GRID = {'book_first_block_off_grid_share': off_grid_share(pr0, TICK), 'trades_off_grid_share': off_grid_share(tpx, TICK)}; print(GRID)
tpt = np.rint(tpx / TICK).astype(np.int64)

incremental_book_L2 sha256 equal to manifest: True
trades sha256 equal to manifest: True
book_snapshot_25 sha256 equal to manifest: True
[sha256_recheck] 7.2 s | peak memory so far 0.2 GB
[load_trades] 1.7 s | peak memory so far 0.49 GB
trades loaded: 2,401,094 | buy 1,142,546 sell 1,258,548 other 0
trades sorted by local ts: True | by exchange ts: True
tick size inferred from the data: 0.01
{'book_first_block_off_grid_share': 0.0, 'trades_off_grid_share': 0.0}


## 3 · Full-depth replay with trades interleaved (one pass)

The book keys are integer ticks (exact equality, no float noise). Trades are evaluated against the book **before** the next message is applied: for each trade the state is the last completed message with `local_timestamp` strictly before the trade. Trades whose price is not explained by the pre-trade book are also compared with the book **after** the next message (to see whether the level appeared and was hit inside the same update interval). For trades at the touch the notebook records whether the touch quantity on the *hit* side shrank at the next message, and — as a control — whether the *opposite* touch shrank.

In [ ]:
class IBook:
    __slots__ = ('bids', 'asks', 'bh', 'ah')
    def __init__(s): s.bids = {}; s.asks = {}; s.bh = []; s.ah = []
    def reset(s): s.bids.clear(); s.asks.clear(); s.bh.clear(); s.ah.clear()
    def upd(s, is_bid, p, a):
        if is_bid:
            d = s.bids
            if a > 0:
                if p not in d: heapq.heappush(s.bh, -p)
                d[p] = a
            else: d.pop(p, None)
        else:
            d = s.asks
            if a > 0:
                if p not in d: heapq.heappush(s.ah, p)
                d[p] = a
            else: d.pop(p, None)
    def best(s):
        bh = s.bh; bd = s.bids
        while bh and -bh[0] not in bd: heapq.heappop(bh)
        ah = s.ah; ad = s.asks
        while ah and ah[0] not in ad: heapq.heappop(ah)
        if len(bh) > 4 * len(bd) + 4000: s.bh = bh = [-p for p in bd]; heapq.heapify(bh)
        if len(ah) > 4 * len(ad) + 4000: s.ah = ah = list(ad); heapq.heapify(ah)
        return (-bh[0] if bh else 0), (ah[0] if ah else 0)          # 0 = side empty

CAT_NAMES = {0: 'no_valid_state', 1: 'through_opposite', 2: 'inside_spread', 3: 'at_touch', 4: 'deeper_1', 5: 'deeper_2_5', 6: 'deeper_6_20', 7: 'deeper_21_100', 8: 'deeper_gt100', 9: 'unknown_side'}
def deeper_code(d): return 4 if d == 1 else 5 if d <= 5 else 6 if d <= 20 else 7 if d <= 100 else 8
TL_ = tL.tolist(); TP_ = tpt.tolist(); TS_ = tside.tolist()

def load_s25_best(stride):
    out = {}; n = 0
    if not HAVE_S25: return out, 0
    rd = pacsv.open_csv(FILES['book_snapshot_25'], read_options=pacsv.ReadOptions(block_size=48 << 20), convert_options=pacsv.ConvertOptions(include_columns=['local_timestamp', 'asks[0].price', 'bids[0].price'], column_types={'local_timestamp': pa.int64(), 'asks[0].price': pa.float64(), 'bids[0].price': pa.float64()}))
    for b in rd:
        L = b.column('local_timestamp').to_numpy(); A = b.column('asks[0].price').to_numpy(); B = b.column('bids[0].price').to_numpy(); idx = np.arange((-n) % stride, len(L), stride); n += len(L)
        At = np.rint(A[idx] / TICK).astype(np.int64).tolist(); Bt = np.rint(B[idx] / TICK).astype(np.int64).tolist()
        for l, a, bb in zip(L[idx].tolist(), At, Bt): out[l] = (a, bb)
    return out, n

def run_pass(max_rows=None, samp=None, verbose=True):
    rd = book_reader(); bk = IBook(); n = NT
    cat = np.zeros(n, np.int8); dd = np.zeros(n, np.int32); rank = np.full(n, -1, np.int16); age = np.full(n, -1, np.int64); kk = np.zeros(n, np.int32); bsz = np.zeros(n, np.int32)
    rest = np.full(n, -1, np.int8); post = np.full(n, -1, np.int8); red_a = np.full(n, -1, np.int8); red_o = np.full(n, -1, np.int8); tq = np.zeros(n, np.float64)
    ML = array('q'); ME = array('q'); MB = array('q'); MA = array('q')
    st = {'rows': 0, 'skipped_pre': 0, 'messages': 0, 'resets': 0, 'crossed': 0, 'locked': 0, 'snap_checked': 0, 'snap_best_equal': 0, 'trades_after_last_message': 0}
    S = {'j': 0, 'L': None, 'bb': 0, 'ba': 0}; pending = []
    def flush(Lcut):
        j = S['j']
        if j >= n or TL_[j] > Lcut: return
        j1 = j
        while j1 < n and TL_[j1] <= Lcut: j1 += 1
        size = j1 - j; bb = S['bb']; ba = S['ba']; cL = S['L']; valid = cL is not None and bb > 0 and ba > 0 and bb < ba; bids = bk.bids; asks = bk.asks
        for t in range(j, j1):
            kk[t] = t - j; bsz[t] = size
            if not valid: continue
            p = TP_[t]; s = TS_[t]; age[t] = TL_[t] - cL
            if s == 2: cat[t] = 9; continue
            if s == 1: dct = bids; touch = bb; opp = ba; dist = bb - p; thr = p >= ba; ins = (p > bb) and (p < ba)
            else: dct = asks; touch = ba; opp = bb; dist = p - ba; thr = p <= bb; ins = (p < ba) and (p > bb)
            tq[t] = dct.get(touch, 0.0)
            if thr: c = 1
            elif ins: c = 2
            elif dist == 0: c = 3
            else: c = deeper_code(dist); dd[t] = dist
            cat[t] = c
            if c <= 2 or c >= 4:
                r = 1 if p in dct else 0; rest[t] = r
                if c >= 4 and r and dist <= 200:
                    occ = 0
                    if s == 1:
                        for q in range(p + 1, bb):
                            if q in bids: occ += 1
                    else:
                        for q in range(ba + 1, p):
                            if q in asks: occ += 1
                    rank[t] = 2 + occ
                if not r: pending.append((t, 0, p, s == 1, 0.0, 0, 0.0))
            else:
                odct = asks if s == 1 else bids; pending.append((t, 3, p, s == 1, dct.get(p, 0.0), opp, odct.get(opp, 0.0)))
        S['j'] = j1
    def end_msg(L, E):
        st['messages'] += 1; bb, ba = bk.best(); ML.append(L); ME.append(E); MB.append(bb); MA.append(ba)
        if bb and ba:
            if bb > ba: st['crossed'] += 1
            elif bb == ba: st['locked'] += 1
        S['L'] = L; S['bb'] = bb; S['ba'] = ba
        if pending:
            bids = bk.bids; asks = bk.asks
            for (t, kind, p, agg_bid, qpre, opp_p, oppq) in pending:
                d = bids if agg_bid else asks; o = asks if agg_bid else bids; post[t] = 1 if p in d else 0
                if kind == 3: red_a[t] = 1 if d.get(p, 0.0) < qpre else 0; red_o[t] = 1 if o.get(opp_p, 0.0) < oppq else 0
            pending.clear()
        if samp is not None and L in samp and bb and ba and bb < ba:
            a0, b0 = samp[L]; st['snap_checked'] += 1; st['snap_best_equal'] += int(a0 == ba and b0 == bb)
    started = False; pL = None; pf = None; pE = None; stop = False; t0 = time.time(); done = 0
    for b in rd:
        Ls = b.column('local_timestamp').to_numpy().tolist(); Es = b.column('timestamp').to_numpy().tolist(); Fs = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool).tolist()
        Bs = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool).tolist(); Ps = np.rint(b.column('price').to_numpy() / TICK).astype(np.int64).tolist(); As = b.column('amount').to_numpy().tolist()
        for i in range(len(Ls)):
            if max_rows is not None and st['rows'] >= max_rows: stop = True; break
            L = Ls[i]; f = Fs[i]; st['rows'] += 1
            if started and (f != pf or ((not f) and L != pL)):
                end_msg(pL, pE); flush(L)
            if not started:
                if not f: st['skipped_pre'] += 1; pL = L; pf = f; pE = Es[i]; continue
                started = True; flush(L)
            if f and (pf is None or not pf): bk.reset(); st['resets'] += 1
            bk.upd(Bs[i], Ps[i], As[i]); pL = L; pf = f; pE = Es[i]
        if stop: break
        if verbose and st['rows'] // 5_000_000 != done // 5_000_000: print(f"   replayed {st['rows']:,} rows | {time.time()-t0:.0f}s | messages {st['messages']:,} | trades evaluated {S['j']:,}")
        done = st['rows']
    if started and not stop:
        end_msg(pL, pE); st['trades_after_last_message'] = n - S['j']; flush(2 ** 62)
    return {'stats': st, 'cat': cat, 'dd': dd, 'rank': rank, 'age': age, 'k': kk, 'bsz': bsz, 'rest': rest, 'post': post, 'red_a': red_a, 'red_o': red_o, 'tq': tq,
            'ML': np.frombuffer(ML, np.int64).copy() if len(ML) else np.zeros(0, np.int64), 'ME': np.frombuffer(ME, np.int64).copy() if len(ME) else np.zeros(0, np.int64), 'MB': np.frombuffer(MB, np.int64).copy() if len(MB) else np.zeros(0, np.int64), 'MA': np.frombuffer(MA, np.int64).copy() if len(MA) else np.zeros(0, np.int64),
            'trades_evaluated': S['j'], 'final_levels': (len(bk.bids), len(bk.asks)), 'seconds': time.time() - t0}

In [ ]:
S25_STRIDE = 40; t0 = time.time(); SAMP, N_S25 = load_s25_best(S25_STRIDE); tick_('load_snapshot25_best_sample', t0); print(f'book_snapshot_25 rows: {N_S25:,} | every {S25_STRIDE}-th row sampled -> {len(SAMP):,} distinct local_timestamps')
t0 = time.time(); PS = run_pass(samp=SAMP); tick_('replay_with_trades', t0)
st = PS['stats']; print(st); print('final levels (bids, asks):', PS['final_levels'], '| trades evaluated:', PS['trades_evaluated'], 'of', NT)
cat = PS['cat']; MLn, MEn, MBn, MAn = PS['ML'], PS['ME'], PS['MB'], PS['MA']; MSG_E_SORTED = bool(np.all(np.diff(MEn) >= 0)) if len(MEn) > 1 else True
print('message exchange timestamps non-decreasing:', MSG_E_SORTED)

[load_snapshot25_best_sample] 4.6 s | peak memory so far 1.24 GB
book_snapshot_25 rows: 790,101 | every 40-th row sampled -> 19,753 distinct local_timestamps
   replayed 5,457,704 rows | 22s | messages 250,025 | trades evaluated 362,871
   replayed 10,247,801 rows | 39s | messages 317,006 | trades evaluated 914,602
   replayed 15,030,188 rows | 55s | messages 463,980 | trades evaluated 1,282,071
   replayed 20,497,640 rows | 71s | messages 605,190 | trades evaluated 1,638,513
   replayed 25,286,884 rows | 86s | messages 690,822 | trades evaluated 1,998,924
   replayed 30,074,828 rows | 100s | messages 817,202 | trades evaluated 2,320,438
[replay_with_trades] 103.6 s | peak memory so far 2.9 GB
{'rows': 31005311, 'skipped_pre': 0, 'messages': 861138, 'resets': 1, 'crossed': 0, 'locked': 0, 'snap_checked': 19753, 'snap_best_equal': 19753, 'trades_after_last_message': 20}
final levels (bids, asks): (26337, 31048) | trades evaluated: 2401094 of 2401094
message exchange timestamps non-decre

## 4 · Mutually exclusive trade categories against the full-depth reference book

State = last completed message with `local_timestamp` strictly before the trade. Each trade gets exactly one category, so every column below sums to 100 %.

In [ ]:
def classify_vec(p, s, bb, ba, valid):
    sell = (s == 1); sd = np.where(sell, bb - p, p - ba); through = np.where(sell, p >= ba, p <= bb)
    c = np.zeros(len(p), np.int8)
    c[sd == 0] = 3; c[sd == 1] = 4; c[(sd >= 2) & (sd <= 5)] = 5; c[(sd >= 6) & (sd <= 20)] = 6; c[(sd >= 21) & (sd <= 100)] = 7; c[sd > 100] = 8
    c[(sd < 0) & ~through] = 2; c[through] = 1; c[s == 2] = 9; c[~valid] = 0
    return c, sd
MEs = np.maximum.accumulate(MEn) if len(MEn) else MEn
def lookup(keys_msg, keys_tr, shift_us):
    idx = np.searchsorted(keys_msg, keys_tr + shift_us, side='left') - 1; ok = idx >= 0; ii = np.where(ok, idx, 0); bb = MBn[ii]; ba = MAn[ii]
    return bb, ba, ok & (bb > 0) & (ba > 0) & (bb < ba)
def state_cat(domain, shift_ms, side=None):
    keys_m, keys_t = (MLn, tL) if domain == 'local' else (MEs, tE); bb, ba, valid = lookup(keys_m, keys_t, int(shift_ms * 1000)); s = tside if side is None else side
    c, sd = classify_vec(tpt, s, bb, ba, valid); return c, sd, valid
cat0, sd0, valid0 = state_cat('local', 0); NV = int(valid0.sum())
tab = {}
for nm, mask in (('all_trades', valid0), ('sell_taker', valid0 & (tside == 1)), ('buy_taker', valid0 & (tside == 0))):
    cnt = np.bincount(cat0[mask], minlength=10); tab[nm + '_count'] = cnt[1:9]; tab[nm + '_share'] = cnt[1:9] / max(cnt[1:9].sum(), 1)
CATTAB = pd.DataFrame(tab, index=[CAT_NAMES[i] for i in range(1, 9)]); display(CATTAB.style.format({c: '{:.4%}' for c in CATTAB.columns if c.endswith('share')}))
SUMS = {c: float(CATTAB[c].sum()) for c in CATTAB.columns if c.endswith('share')}; print('column sums of shares (must be 1.0):', SUMS)
Q = (cat0 >= 5) & (cat0 <= 8); NQ = int(Q.sum()); P_Q = sh(NQ, NV)
print(f'usable trades {NV:,} of {NT:,} | Q (>= 2 ticks deeper than the touch) = {NQ:,} = {P_Q:.4%} of usable trades | through_opposite {sh(int((cat0==1).sum()), NV):.4%} | inside_spread {sh(int((cat0==2).sum()), NV):.4%}')
REPORT['categories'] = {'table': CATTAB.to_dict(), 'usable_trades': NV, 'Q_count': NQ, 'Q_share': P_Q, 'tick': TICK}

,all_trades_count,all_trades_share,sell_taker_count,sell_taker_share,buy_taker_count,buy_taker_share
through_opposite,90359,3.7633%,41675,3.3114%,48684,4.2611%
inside_spread,16500,0.6872%,7663,0.6089%,8837,0.7735%
at_touch,1203782,50.1356%,603087,47.9199%,600695,52.5762%
deeper_1,32786,1.3655%,17354,1.3789%,15432,1.3507%
deeper_2_5,10661,0.4440%,5594,0.4445%,5067,0.4435%
deeper_6_20,26523,1.1046%,13898,1.1043%,12625,1.1050%
deeper_21_100,135427,5.6403%,73870,5.8695%,61557,5.3878%
deeper_gt100,885016,36.8595%,495390,39.3626%,389626,34.1022%


column sums of shares (must be 1.0): {'all_trades_share': 1.0, 'sell_taker_share': 1.0, 'buy_taker_share': 1.0}
usable trades 2,401,054 of 2,401,094 | Q (>= 2 ticks deeper than the touch) = 1,057,627 = 44.0484% of usable trades | through_opposite 3.7633% | inside_spread 0.6872%


## 5 · Is `side` the taker side? (original labels versus swapped labels, and the touch-quantity test)

If `side` is the taker's side, sell trades should print at the best bid or deeper, buy trades at the best ask or deeper. If the labels were swapped, the *swapped* labelling would fit better. A second, independent look: after a trade **at the touch**, the quantity resting at that touch on the hit side should shrink more often than the quantity at the opposite touch (the control).

In [ ]:
sw = tside.copy(); sw[tside == 0] = 1; sw[tside == 1] = 0
cat_sw, _, _ = state_cat('local', 0, side=sw)
def consistent(c, m): return sh(int(((c >= 3) & (c <= 8) & m).sum()), int(m.sum()))
SIDE = {'consistent_share_original_labels(at_touch_or_deeper)': {'all': consistent(cat0, valid0), 'sell': consistent(cat0, valid0 & (tside == 1)), 'buy': consistent(cat0, valid0 & (tside == 0))},
        'consistent_share_swapped_labels': {'all': consistent(cat_sw, valid0), 'sell_labelled_as_buy': consistent(cat_sw, valid0 & (tside == 1)), 'buy_labelled_as_sell': consistent(cat_sw, valid0 & (tside == 0))}}
SIDE['original_minus_swapped'] = SIDE['consistent_share_original_labels(at_touch_or_deeper)']['all'] - SIDE['consistent_share_swapped_labels']['all']
ra, ro = PS['red_a'], PS['red_o']
for nm, m in (('all', (cat0 == 3)), ('sell_taker', (cat0 == 3) & (tside == 1)), ('buy_taker', (cat0 == 3) & (tside == 0))):
    mm = m & (ra >= 0); SIDE['touch_qty_shrank_at_next_message_' + nm] = {'n': int(mm.sum()), 'hit_side_touch_level': sh(int((ra[mm] == 1).sum()), int(mm.sum())), 'opposite_touch_level_(control)': sh(int((ro[mm] == 1).sum()), int(mm.sum()))}
print(json.dumps(SIDE, indent=1, default=jsonable)); REPORT['side_check'] = SIDE

{
 "consistent_share_original_labels(at_touch_or_deeper)": {
  "all": 0.9554949617959446,
  "sell": 0.960797151599762,
  "buy": 0.9496544052067223
 },
 "consistent_share_swapped_labels": {
  "all": 0.037633056149507675,
  "sell_labelled_as_buy": 0.03311400354858164,
  "buy_labelled_as_sell": 0.042610958378956044
 },
 "original_minus_swapped": 0.917861905646437,
 "touch_qty_shrank_at_next_message_all": {
  "n": 1203762,
  "hit_side_touch_level": 0.7989162309493073,
  "opposite_touch_level_(control)": 0.5504809090168987
 },
 "touch_qty_shrank_at_next_message_sell_taker": {
  "n": 603076,
  "hit_side_touch_level": 0.812483003800516,
  "opposite_touch_level_(control)": 0.561377007209705
 },
 "touch_qty_shrank_at_next_message_buy_taker": {
  "n": 600686,
  "hit_side_touch_level": 0.7852954788358644,
  "opposite_touch_level_(control)": 0.5395414576001438
 }
}


## 6 · Timestamp alignment: local clock vs exchange clock, time shifts, and an "oracle" bound

For every shift, the book state is looked up at `trade_time + shift` (local clock: trade `local_timestamp` against message `local_timestamp`; exchange clock: trade `timestamp` against message `timestamp`). If misalignment were the main cause, some shift would make the "deeper ≥ 2 ticks" share collapse. The **oracle** goes further: for each trade it takes the best-fitting state among 13 shifts within ±1 s on either clock, and asks whether the trade is within 1 tick of the hit-side touch at *any* of them. That is deliberately generous; it bounds how much alignment alone could explain.

In [ ]:
SHIFTS_MS = [-2000, -1000, -500, -250, -100, -50, -20, 0, 20, 50, 100, 250, 500, 1000, 2000]; ORACLE_MS = [s for s in SHIFTS_MS if abs(s) <= 1000]
rows = []; BIG = 10 ** 9; minabs = {'local': np.full(NT, BIG, np.int64), 'exchange': np.full(NT, BIG, np.int64)}
t0 = time.time()
for dom in ('local', 'exchange'):
    for s_ms in SHIFTS_MS:
        c, sd, v = state_cat(dom, s_ms); nv = int(v.sum())
        rows.append({'clock': dom, 'shift_ms': s_ms, 'usable': sh(nv, NT), 'at_touch': sh(int((c == 3).sum()), nv), 'deeper_1': sh(int((c == 4).sum()), nv), 'deeper_ge2': sh(int(((c >= 5) & (c <= 8)).sum()), nv), 'inside_spread': sh(int((c == 2).sum()), nv), 'through_opposite': sh(int((c == 1).sum()), nv)})
        if s_ms in ORACLE_MS: ab = np.where(v, np.abs(sd), BIG); minabs[dom] = np.minimum(minabs[dom], ab)
tick_('alignment_scan', t0)
SHIFTTAB = pd.DataFrame(rows); display(SHIFTTAB.style.format({c: '{:.3%}' for c in SHIFTTAB.columns if c not in ('clock', 'shift_ms')}))
best_row = SHIFTTAB.loc[SHIFTTAB.deeper_ge2.idxmin()]; base_row = SHIFTTAB[(SHIFTTAB.clock == 'local') & (SHIFTTAB.shift_ms == 0)].iloc[0]
tm_local = Q & (minabs['local'] <= 1); tm_exch = Q & (minabs['exchange'] <= 1); tm_any = tm_local | tm_exch
ORACLE = {'Q_trades': NQ, 'explained_by_oracle_local_clock': sh(int(tm_local.sum()), NQ), 'explained_by_oracle_exchange_clock': sh(int(tm_exch.sum()), NQ), 'explained_by_oracle_either_clock': sh(int(tm_any.sum()), NQ),
          'share_of_ALL_usable_trades_never_within_1_tick_under_oracle_(either_clock)': sh(int(((np.minimum(minabs['local'], minabs['exchange']) > 1) & valid0).sum()), NV),
          'lowest_deeper_ge2_share_over_all_clock/shift_combinations': {'clock': best_row.clock, 'shift_ms': int(best_row.shift_ms), 'value': float(best_row.deeper_ge2)}, 'deeper_ge2_at_local_shift_0': float(base_row.deeper_ge2)}
LAT = {'median_trade_local_minus_exchange_us': float(np.median(tL - tE)), 'p99_trade_local_minus_exchange_us': float(np.percentile(tL - tE, 99)), 'median_book_local_minus_exchange_us': float(np.median(PS['ML'] - PS['ME'])) if len(MLn) else None, 'p99_book_local_minus_exchange_us': float(np.percentile(PS['ML'] - PS['ME'], 99)) if len(MLn) else None, 'trade_exchange_ts_resolution_ms_share(timestamp%1000==0)': float(np.mean(tE % 1000 == 0)), 'book_message_interval_us_median': float(np.median(np.diff(MLn))) if len(MLn) > 1 else None, 'book_message_interval_us_p90': float(np.percentile(np.diff(MLn), 90)) if len(MLn) > 1 else None}
print(json.dumps({'oracle': ORACLE, 'latency_and_clocks': LAT}, indent=1, default=jsonable)); REPORT['alignment'] = {'shift_table': SHIFTTAB.to_dict('records'), 'oracle': ORACLE, 'latency': LAT, 'message_exchange_ts_non_decreasing': MSG_E_SORTED}

[alignment_scan] 9.4 s | peak memory so far 2.9 GB


,clock,shift_ms,usable,at_touch,deeper_1,deeper_ge2,inside_spread,through_opposite
0,local,-2000,99.998%,14.452%,0.721%,58.770%,0.107%,25.950%
1,local,-1000,99.998%,21.075%,0.904%,57.631%,0.158%,20.233%
2,local,-500,99.998%,28.075%,1.052%,55.469%,0.277%,15.127%
3,local,-250,99.998%,34.404%,1.165%,52.944%,0.359%,11.129%
4,local,-100,99.998%,40.840%,1.207%,50.057%,0.480%,7.416%
5,local,-50,99.998%,43.997%,1.282%,48.344%,0.555%,5.822%
6,local,-20,99.998%,46.650%,1.317%,46.719%,0.613%,4.700%
7,local,0,99.998%,50.136%,1.365%,44.048%,0.687%,3.763%
8,local,20,99.998%,50.519%,1.141%,35.605%,1.666%,11.070%
9,local,50,99.998%,48.564%,0.812%,23.418%,1.989%,25.217%


{
 "oracle": {
  "Q_trades": 1057627,
  "explained_by_oracle_local_clock": 0.15197418371505267,
  "explained_by_oracle_exchange_clock": 0.14958960011421796,
  "explained_by_oracle_either_clock": 0.15290078638310103,
  "share_of_ALL_usable_trades_never_within_1_tick_under_oracle_(either_clock)": 0.39332101652024487,
  "lowest_deeper_ge2_share_over_all_clock/shift_combinations": {
   "clock": "local",
   "shift_ms": 100,
   "value": 0.05859593328596525
  },
  "deeper_ge2_at_local_shift_0": 0.440484470570008
 },
 "latency_and_clocks": {
  "median_trade_local_minus_exchange_us": 4149.0,
  "p99_trade_local_minus_exchange_us": 265223.2999999849,
  "median_book_local_minus_exchange_us": 3709.0,
  "p99_book_local_minus_exchange_us": 6172.0,
  "trade_exchange_ts_resolution_ms_share(timestamp%1000==0)": 1.0,
  "book_message_interval_us_median": 100050.0,
  "book_message_interval_us_p90": 101385.0
 }
}


## 7 · Multi-level sweeps — HEURISTIC grouping only (not verified order identity)

Tardis' CSV has no order IDs. A "possible parent order" is therefore **inferred**: consecutive trades with the **same side**, **consecutive trade IDs**, the **same exchange timestamp** (millisecond resolution), and a price that never improves (a sweep walks away from the touch). Two variants: `G1` as above; `G2` additionally requires the arrival (local) times to lie within 1 ms. Different orders in the same millisecond can be merged by this rule, so group counts are approximate. The grouping is used *only* to ask: **are the deep trades continuation fills of a group that started at the touch?** — and the mechanical check: **was the displayed quantity of the first level actually used up** before the price moved deeper?

In [ ]:
def make_groups(local_gap_us=None):
    brk = np.ones(NT, bool)
    if NT > 1:
        same = (tside[1:] == tside[:-1]) & (tid[1:] == tid[:-1] + 1) & (tE[1:] == tE[:-1])
        if local_gap_us is not None: same &= (tL[1:] - tL[:-1]) <= local_gap_us
        mono = np.where(tside[1:] == 1, tpt[1:] <= tpt[:-1], tpt[1:] >= tpt[:-1]); brk[1:] = ~(same & mono)
    gid = np.cumsum(brk) - 1; first = np.flatnonzero(brk); pos = np.arange(NT) - first[gid]; size = np.bincount(gid)
    chg = np.zeros(NT, bool)
    if NT > 1: chg[1:] = (~brk[1:]) & (tpt[1:] != tpt[:-1])
    nchg = np.bincount(gid, weights=chg).astype(np.int64); multi = nchg[gid] >= 1
    fp = tpt[first][gid]; deeper_than_first = np.where(tside == 1, tpt < fp, tpt > fp)
    cs = np.cumsum(tam); start_cs = (cs - tam)[first]; first_level_qty = np.bincount(gid, weights=np.where(tpt == fp, tam, 0.0)); tq_first = PS['tq'][first]
    mech_ok = (first_level_qty[gid] >= 0.999 * tq_first[gid] - 1e-9)
    return {'gid': gid, 'first': first, 'pos': pos, 'size': size, 'multi': multi, 'cont': (pos > 0) & deeper_than_first, 'mech_ok': mech_ok, 'first_cat': cat0[first][gid]}
GROUPS = {'G1_same_ms': make_groups(None), 'G2_same_ms_and_local<=1ms': make_groups(1000)}
SWEEP = {}
for gname, G in GROUPS.items():
    sz = G['size']; per_trade_size = sz[G['gid']]
    d = {'groups': int(len(sz)), 'trades_per_group': {'1': sh(int((sz == 1).sum()), len(sz)), '2': sh(int((sz == 2).sum()), len(sz)), '3-5': sh(int(((sz >= 3) & (sz <= 5)).sum()), len(sz)), '6-20': sh(int(((sz >= 6) & (sz <= 20)).sum()), len(sz)), '>20': sh(int((sz > 20).sum()), len(sz))},
         'share_of_trades_in_multi_price_groups': sh(int(G['multi'].sum()), NT), 'share_of_groups_that_are_multi_price': sh(int((np.bincount(G['gid'], weights=G['multi'].astype(float)) > 0).sum()), len(sz))}
    firstfill = valid0 & (G['pos'] == 0)
    d['first_fills_(valid)'] = {'n': int(firstfill.sum()), 'at_touch': sh(int(((cat0 == 3) & firstfill).sum()), int(firstfill.sum())), 'deeper_ge2': sh(int((Q & firstfill).sum()), int(firstfill.sum())), 'deeper_1': sh(int(((cat0 == 4) & firstfill).sum()), int(firstfill.sum()))}
    d['Q_decomposition'] = {'Q': NQ, 'first_fill_of_group': sh(int((Q & (G['pos'] == 0)).sum()), NQ), 'non_first_same_price': sh(int((Q & (G['pos'] > 0) & ~G['cont']).sum()), NQ), 'non_first_deeper_than_group_first_price_(sweep_continuation)': sh(int((Q & G['cont']).sum()), NQ)}
    cont_q = Q & G['cont'] & (G['first_cat'] == 3); d['sweep_mechanics_check'] = {'continuation_trades_in_Q_whose_group_started_at_touch': int(cont_q.sum()), 'first_level_displayed_qty_fully_consumed_before_going_deeper': sh(int((cont_q & G['mech_ok']).sum()), int(cont_q.sum()))}
    SWEEP[gname] = d
print(json.dumps(SWEEP, indent=1, default=jsonable)); REPORT['sweep_heuristic'] = {'WARNING': 'heuristic grouping, NOT verified order identity', 'results': SWEEP}

{
 "G1_same_ms": {
  "groups": 1096156,
  "trades_per_group": {
   "1": 0.6909782914110765,
   "2": 0.14013151412755118,
   "3-5": 0.1100244855659231,
   "6-20": 0.051670565138538675,
   ">20": 0.007195143756910513
  },
  "share_of_trades_in_multi_price_groups": 0.41351400653202247,
  "share_of_groups_that_are_multi_price": 0.13167286408138987,
  "first_fills_(valid)": {
   "n": 1096124,
   "at_touch": 0.679435903237225,
   "deeper_ge2": 0.24975185289255594,
   "deeper_1": 0.007176195393951779
  },
  "Q_decomposition": {
   "Q": 1057627,
   "first_fill_of_group": 0.25884267326760757,
   "non_first_same_price": 0.10454063672731502,
   "non_first_deeper_than_group_first_price_(sweep_continuation)": 0.6366166900050774
  },
  "sweep_mechanics_check": {
   "continuation_trades_in_Q_whose_group_started_at_touch": 225314,
   "first_level_displayed_qty_fully_consumed_before_going_deeper": 0.5011583834115945
  }
 },
 "G2_same_ms_and_local<=1ms": {
  "groups": 1172174,
  "trades_per_group": {
  

## 8 · Staleness, yardstick, volatility, and what the book says about the deep trades

In [ ]:
age = PS['age']; kk = PS['k']; bsz = PS['bsz']; v = valid0 & (age >= 0)
def grp_table(values, edges, labels):
    b = np.digitize(values, edges[1:-1]); out = []
    for i, lab in enumerate(labels):
        m = v & (b == i); n_ = int(m.sum()); out.append({'bin': lab, 'trades': n_, 'at_touch': sh(int(((cat0 == 3) & m).sum()), n_), 'deeper_ge2_(Q)': sh(int((Q & m).sum()), n_)})
    return pd.DataFrame(out)
AGE_TAB = grp_table(age, [0, 10_000, 25_000, 50_000, 75_000, 100_000, 10 ** 15], ['<10ms', '10-25ms', '25-50ms', '50-75ms', '75-100ms', '>=100ms'])
K_TAB = grp_table(kk, [0, 1, 2, 4, 10, 10 ** 9], ['1st trade after update', '2nd', '3rd-4th', '5th-10th', '11th+'])
B_TAB = grp_table(bsz, [1, 2, 4, 11, 10 ** 9, 10 ** 10], ['1 trade in interval', '2-3', '4-10', '>10', '(unused)'])[:4]
for nm, tb in (('by age of the book state at the trade', AGE_TAB), ('by position of the trade inside a book-update interval', K_TAB), ('by number of trades in the interval', B_TAB)): print(nm); display(tb.style.format({'at_touch': '{:.3%}', 'deeper_ge2_(Q)': '{:.3%}'}))
lo = v & (age < 10_000); hi = v & (age >= 50_000); STALE_RATIO = sh(sh(int((Q & hi).sum()), int(hi.sum())), sh(int((Q & lo).sum()), int(lo.sum())))
first_in_int = v & (kk == 0); later_in_int = v & (kk >= 1)
STALE = {'P(Q|age>=50ms)/P(Q|age<10ms)': STALE_RATIO, 'P(Q|first_trade_in_interval)': sh(int((Q & first_in_int).sum()), int(first_in_int.sum())), 'P(Q|later_trade_in_interval)': sh(int((Q & later_in_int).sum()), int(later_in_int.sum())), 'age_median_us': float(np.median(age[v])) if v.any() else None}
print(STALE)
# yardstick: distance in ticks and bps for the deep trades
sdq = sd0[Q]; bpsq = sdq / np.maximum(tpt[Q], 1) * 1e4
YARD = {'tick_in_bps_at_median_price': TICK / float(np.median(tpx)) * 1e4, 'Q_distance_ticks_quantiles(50,75,90,95,99)': [float(x) for x in np.percentile(sdq, [50, 75, 90, 95, 99])] if NQ else None, 'Q_distance_bps_quantiles(50,75,90,95,99)': [float(x) for x in np.percentile(bpsq, [50, 75, 90, 95, 99])] if NQ else None,
        'Q_share_within_bps': {str(x): sh(int((bpsq <= x).sum()), NQ) for x in (0.01, 0.1, 0.5, 1, 2, 5)}}
print(json.dumps(YARD, indent=1, default=jsonable))
# time of day and volatility
hour = np.clip((tL - DAY0_US) // 3_600_000_000, 0, 23).astype(int); HOUR = pd.DataFrame({'h': hour[v], 'q': Q[v]}).groupby('h').q.agg(['mean', 'size']).rename(columns={'mean': 'deeper_ge2_share', 'size': 'trades'})
mn = np.clip((tL - DAY0_US) // 60_000_000, 0, 1439).astype(int); dfm = pd.DataFrame({'m': mn, 'p': tpx}); g = dfm.groupby('m').p; rng_bps = ((g.max() - g.min()) / g.mean() * 1e4).rename('rng_bps')
dfv = pd.DataFrame({'m': mn[v], 'q': Q[v]}).join(rng_bps, on='m'); dfv['vol_quintile'] = pd.qcut(dfv.rng_bps.rank(method='first'), 5, labels=['Q1 calmest', 'Q2', 'Q3', 'Q4', 'Q5 most volatile']); VOL = dfv.groupby('vol_quintile', observed=True).agg(deeper_ge2_share=('q', 'mean'), trades=('q', 'size'), minute_range_bps_median=('rng_bps', 'median'))
display(HOUR.style.format({'deeper_ge2_share': '{:.3%}'})); display(VOL.style.format({'deeper_ge2_share': '{:.3%}', 'minute_range_bps_median': '{:.2f}'}))
# what the book says about the deep trades
rest = PS['rest']; post = PS['post']; rk = PS['rank']
BOOKSAYS = {'Q_trades': NQ, 'price_is_a_resting_level_in_pre_trade_book': sh(int((Q & (rest == 1)).sum()), NQ), 'price_not_resting_pre_but_present_after_next_message': sh(int((Q & (rest == 0) & (post == 1)).sum()), NQ), 'price_not_resting_pre_and_absent_after': sh(int((Q & (rest == 0) & (post == 0)).sum()), NQ),
            'rank_of_resting_level_quantiles(50,75,90,99)': [float(x) for x in np.percentile(rk[Q & (rk > 0)], [50, 75, 90, 99])] if int((Q & (rk > 0)).sum()) else None,
            'inside_spread_trades_present_after_next_message': sh(int(((cat0 == 2) & (post == 1)).sum()), int((cat0 == 2).sum())), 'through_opposite_trades_present_after_next_message': sh(int(((cat0 == 1) & (post == 1)).sum()), int((cat0 == 1).sum()))}
print(json.dumps(BOOKSAYS, indent=1, default=jsonable))
REPORT['staleness'] = {'age_table': AGE_TAB.to_dict('records'), 'position_table': K_TAB.to_dict('records'), 'interval_size_table': B_TAB.to_dict('records'), 'summary': STALE}; REPORT['yardstick'] = YARD; REPORT['time_of_day'] = HOUR.reset_index().to_dict('records'); REPORT['volatility'] = VOL.reset_index().to_dict('records'); REPORT['book_says_about_Q'] = BOOKSAYS

by age of the book state at the trade


,bin,trades,at_touch,deeper_ge2_(Q)
0,<10ms,274427,59.598%,36.527%
1,10-25ms,357875,55.297%,40.326%
2,25-50ms,584434,50.999%,43.485%
3,50-75ms,596782,47.173%,46.361%
4,75-100ms,577806,44.892%,47.857%
5,>=100ms,9730,34.635%,58.921%


by position of the trade inside a book-update interval


,bin,trades,at_touch,deeper_ge2_(Q)
0,1st trade after update,407091,96.728%,2.309%
1,2nd,223716,85.493%,11.537%
2,3rd-4th,274824,68.865%,25.772%
3,5th-10th,467209,46.268%,46.382%
4,11th+,1028214,20.747%,71.472%


by number of trades in the interval


,bin,trades,at_touch,deeper_ge2_(Q)
0,1 trade in interval,183375,98.999%,0.627%
1,2-3,235332,93.530%,4.713%
2,4-10,424903,64.245%,29.532%
3,>10,1557444,33.976%,59.065%


{'P(Q|age>=50ms)/P(Q|age<10ms)': 1.2920215606353507, 'P(Q|first_trade_in_interval)': 0.023090660319191533, 'P(Q|later_trade_in_interval)': 0.5257003264353451, 'age_median_us': 49310.0}
{
 "tick_in_bps_at_median_price": 0.0017322298336401114,
 "Q_distance_ticks_quantiles(50,75,90,95,99)": [
  375.0,
  832.0,
  1630.0,
  2401.0,
  5829.0
 ],
 "Q_distance_bps_quantiles(50,75,90,95,99)": [
  0.6476203799941865,
  1.4327665910806011,
  2.813168357104431,
  4.142202189611159,
  10.04726092048859
 ],
 "Q_share_within_bps": {
  "0.01": 0.010281507563630657,
  "0.1": 0.08797903230534017,
  "0.5": 0.4171423384614803,
  "1": 0.6400555205190488,
  "2": 0.8343064237202719,
  "5": 0.9648704127258476
 }
}


,deeper_ge2_share,trades
h,,
0,41.834%,63852
1,41.668%,69967
2,40.997%,66991
3,33.471%,34307
4,28.479%,30988
5,35.176%,41352
6,39.345%,56142
7,49.094%,340170
8,42.414%,228598


,deeper_ge2_share,trades,minute_range_bps_median
vol_quintile,,,
Q1 calmest,32.844%,480211,7.29
Q2,41.122%,480211,12.80
Q3,44.937%,480210,18.49
Q4,48.476%,480211,29.13
Q5 most volatile,52.863%,480211,53.62


{
 "Q_trades": 1057627,
 "price_is_a_resting_level_in_pre_trade_book": 0.8892312696253026,
 "price_not_resting_pre_but_present_after_next_message": 0.020762518354769686,
 "price_not_resting_pre_and_absent_after": 0.09000621201992763,
 "rank_of_resting_level_quantiles(50,75,90,99)": [
  4.0,
  6.0,
  8.0,
  27.0
 ],
 "inside_spread_trades_present_after_next_message": 0.38103030303030305,
 "through_opposite_trades_present_after_next_message": 0.7093150654611051
}


## 9 · What the data supports — explanations compared on equal footing

Each row asks: *of the deep trades (Q), how many does this explanation account for?* The explanations overlap, so the union and the **residual** (not explained by timing or sweeps) are shown. Grades follow the cut-offs frozen in the introduction; they describe the evidence, they are not significance tests.

In [ ]:
Gmain = GROUPS['G1_same_ms']; sw_cont = Q & Gmain['cont']; sw_cont2 = Q & GROUPS['G2_same_ms_and_local<=1ms']['cont']
union = tm_any | sw_cont; resid = Q & ~union
ATTR = {'Q_trades': NQ, 'timing_oracle_local': sh(int(tm_local.sum()), NQ), 'timing_oracle_exchange': sh(int(tm_exch.sum()), NQ), 'timing_oracle_either': sh(int(tm_any.sum()), NQ), 'sweep_continuation_G1': sh(int(sw_cont.sum()), NQ), 'sweep_continuation_G2': sh(int(sw_cont2.sum()), NQ),
        'timing_AND_sweep_both': sh(int((tm_any & sw_cont).sum()), NQ), 'timing_only': sh(int((tm_any & ~sw_cont).sum()), NQ), 'sweep_only': sh(int((sw_cont & ~tm_any).sum()), NQ), 'union_timing_or_sweep': sh(int(union.sum()), NQ), 'RESIDUAL_not_explained_by_timing_or_sweep': sh(int(resid.sum()), NQ), 'residual_count': int(resid.sum()),
        'residual_share_of_all_usable_trades': sh(int(resid.sum()), NV)}
def grade3(x): return 'n/a' if x != x else ('SUPPORTED' if x >= 0.5 else 'PARTIAL' if x >= 0.2 else 'NOT SUPPORTED')
within1bp = YARD['Q_share_within_bps']['1'] if NQ else float('nan')
THROUGH = sh(int((cat0 == 1).sum()), NV)
S25_OK = (st['snap_checked'] >= min(5000, len(SAMP) // 2) and st['snap_best_equal'] / max(st['snap_checked'], 1) >= 0.999) if HAVE_S25 else None
GRADES = {'H-timing': grade3(ATTR['timing_oracle_either']), 'H-sweep': grade3(ATTR['sweep_continuation_G1']),
          'H-stale': ('n/a' if STALE_RATIO != STALE_RATIO else 'SUPPORTED' if STALE_RATIO >= 1.5 else 'NOT SUPPORTED' if STALE_RATIO <= 1.1 else 'PARTIAL'),
          'H-tolerance': ('1 tick is too tight a yardstick' if within1bp >= 0.9 else 'NOT the main factor (deep trades are not all within 1 bp)') if within1bp == within1bp else 'n/a',
          'H-side': 'SUPPORTED (labels behave as taker side)' if SIDE['original_minus_swapped'] >= 0.20 else 'NOT SUPPORTED', 'H-reconstruction': ('SUPPORTED (book suspect)' if (S25_OK is False or THROUGH > 0.01) else 'NOT SUPPORTED by these checks') if S25_OK is not None else 'UNTESTED (no book_snapshot_25)'}
print(json.dumps({'attribution_of_Q': ATTR, 'grades': GRADES, 'through_opposite_share_of_usable': THROUGH, 'best_bid_ask_recheck_vs_book_snapshot_25': {'checked': st['snap_checked'], 'equal_share': sh(st['snap_best_equal'], st['snap_checked'])}}, indent=1, default=jsonable))
REPORT['attribution'] = ATTR; REPORT['grades'] = GRADES

{
 "attribution_of_Q": {
  "Q_trades": 1057627,
  "timing_oracle_local": 0.15197418371505267,
  "timing_oracle_exchange": 0.14958960011421796,
  "timing_oracle_either": 0.15290078638310103,
  "sweep_continuation_G1": 0.6366166900050774,
  "sweep_continuation_G2": 0.6022964617960774,
  "timing_AND_sweep_both": 0.04943330682745429,
  "timing_only": 0.10346747955564675,
  "sweep_only": 0.5871833831776231,
  "union_timing_or_sweep": 0.7400841695607242,
  "RESIDUAL_not_explained_by_timing_or_sweep": 0.25991583043927585,
  "residual_count": 274894,
  "residual_share_of_all_usable_trades": 0.11448888696380839
 },
 "grades": {
  "H-timing": "NOT SUPPORTED",
  "H-sweep": "SUPPORTED",
  "H-stale": "PARTIAL",
  "H-tolerance": "NOT the main factor (deep trades are not all within 1 bp)",
  "H-side": "SUPPORTED (labels behave as taker side)",
  "H-reconstruction": "SUPPORTED (book suspect)"
 },
 "through_opposite_share_of_usable": 0.037633056149507675,
 "best_bid_ask_recheck_vs_book_snapshot_25": {


## 10 · Execution tests (E) and input-data integrity tests (D) — kept apart, and neither says anything about whether trades "fit" the book

In [ ]:
TESTS = {'E': [], 'D': []}
def tcase(group, name, fn):
    try: ok, msg = fn()
    except Exception as e: ok, msg = False, f'EXC {type(e).__name__}: {str(e)[:200]}'
    TESTS[group].append((name, bool(ok), msg)); print(('PASS ' if ok else 'FAIL ') + name + ' | ' + str(msg))
# ---------- E: notebook execution
def e01():
    now = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file() and not str(p.relative_to(BASE)).startswith(STAGE.name)}
    bad = [k for k in PIN0 if now.get(k) != PIN0[k]]; new = [k for k in now if k not in PIN0]; return (not bad and not new), (f'{len(PIN0)} pre-existing files (incl. all of notebook 07 output) unchanged; nothing new outside staging' if not bad and not new else f'CHANGED {bad[:3]} NEW {new[:3]}')
def e02():
    cases = [(1, 100, 101, 100, 3), (1, 100, 101, 101, 1), (1, 100, 101, 99, 4), (1, 100, 101, 98, 5), (1, 100, 101, 94, 6), (1, 100, 101, 79, 7), (1, 100, 101, -5, 8), (0, 100, 101, 101, 3), (0, 100, 101, 100, 1), (0, 100, 101, 102, 4), (0, 100, 101, 103, 5), (1, 100, 105, 102, 2), (0, 100, 105, 103, 2), (1, 100, 105, 105, 1), (0, 100, 105, 100, 1), (2, 100, 101, 100, 9)]
    got = [int(classify_vec(np.array([p]), np.array([s], np.int8), np.array([bb]), np.array([ba]), np.array([True]))[0][0]) for s, bb, ba, p, _ in cases]; bad = [(c, g) for c, g in zip(cases, got) if g != c[4]]
    return not bad, f'{len(cases)} hand-made cases (tight and wide spread, both sides) classified as expected' if not bad else f'WRONG {bad[:3]}'
def e03():
    ok_sum = all(abs(v_ - 1.0) < 1e-9 for v_ in SUMS.values()); c1, _ = classify_vec(np.array([101]), np.array([0], np.int8), np.array([100]), np.array([101]), np.array([True]))
    return ok_sum and int(c1[0]) == 3, f'category shares sum to 1 for all/sell/buy {SUMS}; old overlap case (one-tick spread, buy at the ask) falls in exactly one class (at_touch)'
def e04():
    n_ = PS['trades_evaluated']; same = bool(np.array_equal(PS['cat'][:n_], cat0[:n_])); return same and n_ == NT, f'in-pass (loop) categories == independent vectorised categories for {n_:,}/{NT:,} trades'
def e05():
    m_ok = st['messages'] == SRC_REPORT['coverage']['messages']; r_ok = st['rows'] == SRC_REPORT['book_scan']['rows']; c_ok = st['crossed'] == SRC_REPORT['crossed']['unpruned']['crossed_messages'] and st['locked'] == SRC_REPORT['crossed']['unpruned']['locked_messages']
    mm = re.search(r'unpruned: messages [\d,]+ .*?final levels \((\d+), (\d+)\)', SRC_SHARE); f_ok = bool(mm) and PS['final_levels'] == (int(mm.group(1)), int(mm.group(2)))
    return m_ok and r_ok and c_ok and f_ok, f"replay == notebook 07 record: messages {st['messages']:,}={SRC_REPORT['coverage']['messages']:,} {m_ok}, rows {r_ok}, crossed/locked {c_ok}, final levels {PS['final_levels']} vs {mm.groups() if mm else None} {f_ok}"
def e06():
    a = run_pass(max_rows=300_000, verbose=False); b = run_pass(max_rows=300_000, verbose=False); dig = lambda r: hashlib.sha256(r['cat'].tobytes() + r['age'].tobytes() + r['rest'].tobytes() + r['ML'].tobytes() + r['MB'].tobytes() + r['MA'].tobytes()).hexdigest()[:16]
    return dig(a) == dig(b) and a['stats']['messages'] == b['stats']['messages'], f"two 300k-row runs identical (digest {dig(a)}, {a['stats']['messages']:,} messages, {a['trades_evaluated']:,} trades evaluated)"
def e07():
    m = PS['cat'] > 0; a_ok = bool(np.all(PS['age'][m] > 0)); idx = np.searchsorted(MLn, tL, side='left') - 1; c_ok = bool(np.all(MLn[idx[idx >= 0]] < tL[idx >= 0]))
    return a_ok and c_ok, 'every trade was judged against a message strictly BEFORE it (age > 0 for all; state timestamp < trade timestamp for all) - no look-ahead'
def e08():
    bad = []
    for gname, G in GROUPS.items():
        g = G['gid']; ok = int(G['size'].sum()) == NT and int((G['pos'] == 0).sum()) == len(G['size']); idx = np.flatnonzero(G['pos'] > 0)
        ok &= bool(np.all(tside[idx] == tside[idx - 1])) and bool(np.all(tid[idx] == tid[idx - 1] + 1)) and bool(np.all(tE[idx] == tE[idx - 1])) and bool(np.all(np.where(tside[idx] == 1, tpt[idx] <= tpt[idx - 1], tpt[idx] >= tpt[idx - 1])))
        if not ok: bad.append(gname)
    lab = REPORT.get('sweep_heuristic', {}).get('WARNING', ''); return (not bad) and 'NOT verified order identity' in lab, f'groups partition all {NT:,} trades; members share side/exchange-ts, have consecutive IDs and non-improving prices; report labelled heuristic' if not bad else f'invariant broken {bad}'
def e09():
    if not HAVE_S25: return True, 'book_snapshot_25 not present -> best bid/ask re-check not possible (skipped)'
    n_ = st['snap_checked']; r = sh(st['snap_best_equal'], n_); need = min(5000, len(SAMP) // 2); return n_ >= need and r >= 0.999, f'best bid & ask equal to book_snapshot_25 at {n_:,} aligned checkpoints: {r:.4%} (need >= 99.9% on >= {need:,} checkpoints; same-provider consistency only)'
for nm, fn in (('E01 nothing pre-existing modified (read-only)', e01), ('E02 classifier correct on hand-made cases', e02), ('E03 categories mutually exclusive, sums = 100%, old overlap case fixed', e03), ('E04 loop and vectorised classifiers agree', e04), ('E05 replay reproduces notebook 07 numbers', e05), ('E06 replay deterministic', e06), ('E07 no look-ahead in the book state', e07), ('E08 heuristic group invariants', e08), ('E09 best bid/ask re-check vs book_snapshot_25', e09)): tcase('E', nm, fn)
# ---------- D: input data integrity (re-check of the saved files)
def d01(): bad = [k for k, v_ in SHA_CHECK.items() if not v_['equal']]; return bool(SHA_CHECK) and not bad, ('SHA-256 of every saved raw file equals the value recorded by notebook 07' if not bad else f'MISMATCH {bad}')
def d02(): a = NT == SRC_REPORT['trades_scan']['rows']; b = st['rows'] == SRC_REPORT['book_scan']['rows']; return a and b, f"row counts equal notebook 07: trades {NT:,} {a}, book {st['rows']:,} {b}"
def d03(): return GRID['book_first_block_off_grid_share'] <= 1e-4 and GRID['trades_off_grid_share'] <= 1e-4, f'prices lie on the {TICK} tick grid: {GRID}'
def d04(): ok = bool(np.all(tpx > 0)) and bool(np.all(tam > 0)) and sh(int((tside == 2).sum()), NT) <= 0.01 and bool(np.all(np.isfinite(tpx))); return ok, f'trade price>0, amount>0, finite; unknown-side share {sh(int((tside == 2).sum()), NT):.4%}'
def d05(): dif = np.diff(tid); return bool(np.all(dif > 0)), f'trade IDs strictly increasing; gaps(>1): {int((dif > 1).sum())}; duplicates/backwards: {int((dif <= 0).sum())}'
def d06(): inside = float(np.mean((tL >= DAY0_US) & (tL < DAY1_US))); return inside >= 0.999, f'trade local_timestamp inside {DAY} UTC: {inside:.4%}'
def d07(): dm = np.diff(MLn); return bool(np.all(dm > 0)) and len(MLn) > 1000, f'book message local timestamps strictly increasing ({len(MLn):,} messages)'
for nm, fn in (('D01 saved raw files match notebook 07 SHA-256', d01), ('D02 row counts match notebook 07', d02), ('D03 prices on tick grid', d03), ('D04 trade values sane', d04), ('D05 trade IDs increasing', d05), ('D06 trades inside the day', d06), ('D07 book messages strictly ordered', d07)): tcase('D', nm, fn)
OBS = {'trade_exchange_ts_non_decreasing': TRADES_SORTED_EXCH, 'message_exchange_ts_non_decreasing': MSG_E_SORTED, 'trade_id_gaps(>1)': int((np.diff(tid) > 1).sum()), 'unknown_side_trades': int((tside == 2).sum()), 'trades_before_first_book_message': int((tL <= (MLn[0] if len(MLn) else 0)).sum()), 'trades_after_last_message': st['trades_after_last_message']}
print('\nObservations (not pass/fail):', OBS); REPORT['observations'] = OBS
E_OK = all(t[1] for t in TESTS['E']); D_OK = all(t[1] for t in TESTS['D'])
print(f"\nE (execution): {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} | D (input integrity): {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])}")

PASS E01 nothing pre-existing modified (read-only) | 759 pre-existing files (incl. all of notebook 07 output) unchanged; nothing new outside staging
PASS E02 classifier correct on hand-made cases | 16 hand-made cases (tight and wide spread, both sides) classified as expected
PASS E03 categories mutually exclusive, sums = 100%, old overlap case fixed | category shares sum to 1 for all/sell/buy {'all_trades_share': 1.0, 'sell_taker_share': 1.0, 'buy_taker_share': 1.0}; old overlap case (one-tick spread, buy at the ask) falls in exactly one class (at_touch)
PASS E04 loop and vectorised classifiers agree | in-pass (loop) categories == independent vectorised categories for 2,401,094/2,401,094 trades
PASS E05 replay reproduces notebook 07 numbers | replay == notebook 07 record: messages 861,138=861,138 True, rows True, crossed/locked True, final levels (26337, 31048) vs ('26337', '31048') True
PASS E06 replay deterministic | two 300k-row runs identical (digest 821da1631a55f6d8, 11,959 messag

## 11 · Recommendation (rule frozen in the introduction) and the FINAL TRADE DIAGNOSTIC SHARE BLOCK

In [ ]:
resid_share = ATTR['RESIDUAL_not_explained_by_timing_or_sweep']
integrity_ok = E_OK and D_OK and (S25_OK is not False) and GRADES['H-side'].startswith('SUPPORTED') and THROUGH <= 0.01
if not integrity_ok: REC = 'HOLD - do not download more days yet: at least one frozen condition failed (see E/D tests, best bid/ask re-check, side check, or through-opposite share).'
else:
    REC = 'PROCEED - ask for approval of the remaining free days (BTCUSDT and ETHUSDT on the 1st of each month in the study window): the files are intact, the reference book reproduces the earlier audit, and the side labels behave as taker side.'
    if resid_share == resid_share and resid_share > 0.20: REC += f' CAVEAT: {resid_share:.1%} of the deep trades are explained by neither timing nor sweeps; do NOT use trades for exact price-level alignment with the book (use them only as flow aggregates such as signed volume) until that residual is understood or raised with Tardis.'
    elif NQ == 0: REC += ' (No deep trades were found.)'
print(REC)
def pct(x, d=2): return 'nan' if x != x else f'{x:.{d}%}'
unc = ['One symbol and one day. Nothing here shows the same behaviour on other days or on ETHUSDT.',
       'Heuristic order grouping is NOT verified order identity: Tardis CSVs carry no order IDs; the exchange timestamp has millisecond resolution, so different orders in the same millisecond can be merged.',
       'The book is the provider\'s own reconstruction of the same feed; best bid/ask agreement with book_snapshot_25 is consistency, not independent validation.',
       'The evidence grades are descriptive cut-offs fixed in advance, not statistical tests; overlapping explanations cannot be separated causally from this data alone.',
       'The depth feed is batched (about 10 messages/s) and the trade feed is not, so some trade-to-book mismatch is expected and cannot be repaired without order-level data.',
       'Publication of results is not addressed in the Tardis terms: ask Tardis in writing. Raw files, per-trade tables derived from them, and trained models must not be given to third parties.']
L = []; P = L.append
P('=' * 100); P('FINAL TRADE DIAGNOSTIC SHARE BLOCK - Tardis.dev Binance Spot BTCUSDT 2024-05-01 (read-only follow-up to notebook 07)'); P('=' * 100)
P(f'run: {RUN_STARTED} | source (read-only): {SRC} | output: {OUT_FINAL} | network used: NO | downloads: NONE')
P('Scope: reference = FULL-DEPTH (unpruned) replay. Categories are mutually exclusive. Previous audit thresholds/results untouched. Not a validation of anything: it describes how trades relate to the rebuilt book.')
P('\n[A] INPUTS'); [P(f"  {v_['file']}: sha256 equal to notebook 07 manifest: {v_['equal']}") for v_ in SHA_CHECK.values()]; P(f'  trades {NT:,} | book rows {st["rows"]:,} | tick {TICK} | grid check {GRID} | usable trades (valid pre-trade state) {NV:,}')
P('\n[B] REFERENCE REPLAY'); P(f"  messages {st['messages']:,} | crossed {st['crossed']} | locked {st['locked']} | resets {st['resets']} | final levels {PS['final_levels']} | best bid/ask equal to book_snapshot_25 at {st['snap_checked']:,} aligned checkpoints: {pct(sh(st['snap_best_equal'], st['snap_checked']), 4)} (same provider: consistency only)")
P('\n[C] MUTUALLY EXCLUSIVE TRADE CATEGORIES (state strictly before the trade, local clock)'); P(CATTAB.assign(**{c: CATTAB[c].map('{:.3%}'.format) for c in CATTAB.columns if c.endswith('share')}).to_string())
P(f'  Q (>=2 ticks deeper than the hit-side touch) = {NQ:,} trades = {pct(P_Q)} of usable | through_opposite {pct(THROUGH)} | inside_spread {pct(sh(int((cat0==2).sum()), NV))} | 1 tick = {YARD["tick_in_bps_at_median_price"]:.4f} bps at the median price')
P('\n[D] TAKER-SIDE CHECK'); P('  ' + json.dumps(SIDE, default=jsonable)); P(f'  grade H-side: {GRADES["H-side"]}')
P('\n[E] TIMESTAMP ALIGNMENT'); P(SHIFTTAB.assign(**{c: SHIFTTAB[c].map('{:.3%}'.format) for c in SHIFTTAB.columns if c not in ('clock', 'shift_ms')}).to_string(index=False)); P('  oracle: ' + json.dumps(ORACLE, default=jsonable)); P('  latency/clocks: ' + json.dumps(LAT, default=jsonable)); P(f'  grade H-timing (oracle explains {pct(ATTR["timing_oracle_either"])} of Q): {GRADES["H-timing"]}')
P('\n[F] MULTI-LEVEL SWEEPS - HEURISTIC GROUPING, NOT VERIFIED ORDER IDENTITY'); P('  ' + json.dumps(SWEEP, default=jsonable)); P(f'  grade H-sweep (continuation fills = {pct(ATTR["sweep_continuation_G1"])} of Q): {GRADES["H-sweep"]}')
P('\n[G] STALENESS OF THE BOOK STATE'); P(AGE_TAB.to_string(index=False)); P(K_TAB.to_string(index=False)); P(B_TAB.to_string(index=False)); P('  ' + json.dumps(STALE, default=jsonable)); P(f'  grade H-stale: {GRADES["H-stale"]}')
P('\n[H] YARDSTICK, TIME OF DAY, VOLATILITY'); P('  ' + json.dumps(YARD, default=jsonable)); P(f'  grade H-tolerance: {GRADES["H-tolerance"]}'); P(HOUR.assign(deeper_ge2_share=HOUR.deeper_ge2_share.map('{:.2%}'.format)).T.to_string()); P(VOL.assign(deeper_ge2_share=VOL.deeper_ge2_share.map('{:.2%}'.format)).to_string())
P('\n[I] WHAT THE BOOK SAYS ABOUT THE DEEP TRADES'); P('  ' + json.dumps(BOOKSAYS, default=jsonable))
P('\n[J] ATTRIBUTION OF THE DEEP TRADES AND RESIDUAL'); P('  ' + json.dumps(ATTR, default=jsonable)); P('  GRADES: ' + json.dumps(GRADES, default=jsonable))
P('\n' + '-' * 100); P('[E] NOTEBOOK EXECUTION TESTS'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['E']]; P(f"  => E: {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])}")
P('[D] INPUT-DATA INTEGRITY TESTS'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['D']]; P(f"  => D: {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])}"); P('  observations (not pass/fail): ' + json.dumps(OBS, default=jsonable))
P('  Passing E and D means the code and the files are sound; it does NOT mean trades fit the book or that the book is validated.')
REQ = {'stage_seconds': STAGES, 'peak_memory_gb': round(peak_mem_gb(), 2), 'replay_seconds': round(PS['seconds'], 1)}; P('\n[K] PROCESSING'); P('  ' + json.dumps(REQ, default=jsonable))
P('\nRECOMMENDATION (rule-based, frozen in advance; nothing is downloaded without your approval):'); P('  ' + REC)
P('\nUNCERTAINTIES'); [P('  - ' + u) for u in unc]
P('\nNothing was downloaded. No models, graphs or synthetic scenarios were created. Notebook 07 outputs and all earlier artifacts were not modified. WAITING FOR YOUR APPROVAL before the next stage.'); P('=' * 100)
SHARE = '\n'.join(L)
STAGE.mkdir(exist_ok=True); (STAGE / 'reports').mkdir(exist_ok=True); REPORT['tests'] = TESTS; REPORT['recommendation'] = REC; REPORT['uncertainties'] = unc; REPORT['finished'] = UTC(); REPORT['processing'] = REQ
dump_json(REPORT, STAGE / 'reports' / 'trade_diagnostic_report.json'); (STAGE / 'reports' / 'share_block.txt').write_text(SHARE, encoding='utf-8')
CATTAB.to_csv(STAGE / 'reports' / 'category_table.csv'); SHIFTTAB.to_csv(STAGE / 'reports' / 'alignment_shift_table.csv', index=False); AGE_TAB.to_csv(STAGE / 'reports' / 'age_table.csv', index=False); HOUR.to_csv(STAGE / 'reports' / 'hour_table.csv'); VOL.to_csv(STAGE / 'reports' / 'volatility_table.csv')
G1 = GROUPS['G1_same_ms']
PT = pd.DataFrame({'trade_id': tid, 'side_code(0buy,1sell,2other)': tside, 'price_ticks': tpt, 'local_ts_us': tL, 'exchange_ts_us': tE, 'category_code': cat0, 'ticks_deeper_than_touch': np.clip(sd0, -10 ** 6, 10 ** 6).astype(np.int32), 'state_age_us': np.clip(age, -1, 2 ** 31 - 1).astype(np.int32), 'pos_in_update_interval': kk, 'trades_in_interval': bsz, 'price_is_resting_level_pre': rest, 'price_present_after_next_msg': post, 'level_rank_if_resting': PS['rank'],
                   'heuristic_group_G1_id': G1['gid'].astype(np.int32), 'heuristic_group_G1_pos': np.minimum(G1['pos'], 32000).astype(np.int16), 'heuristic_group_G1_multi_price': G1['multi']})
PT.to_parquet(STAGE / 'reports' / 'per_trade_diagnostic.parquet', index=False)
MAN2 = {str(p.relative_to(STAGE)): sha256_file(p) for p in sorted(STAGE.rglob('*')) if p.is_file()}; dump_json(MAN2, STAGE / 'manifest.json')
okm = all(sha256_file(STAGE / k) == v_ for k, v_ in MAN2.items()); all_ok = E_OK and D_OK and okm
if all_ok: os.rename(STAGE, OUT_FINAL); print('\nSAVED ->', OUT_FINAL, '(diagnostic results; per-trade table is derived from licensed data - keep private)')
else: print('\n!! E or D tests FAILED (or manifest mismatch) -> staging kept at', STAGE, '(not renamed). Share this block anyway.')
print(SHARE); print(f'manifest verified: {okm} ({len(MAN2)} files)')

HOLD - do not download more days yet: at least one frozen condition failed (see E/D tests, best bid/ask re-check, side check, or through-opposite share).

SAVED -> /content/drive/MyDrive/research datasets/TGNN_Tardis_TradeDiag_BTCUSDT_2024-05-01_v1 (diagnostic results; per-trade table is derived from licensed data - keep private)
FINAL TRADE DIAGNOSTIC SHARE BLOCK - Tardis.dev Binance Spot BTCUSDT 2024-05-01 (read-only follow-up to notebook 07)
run: 2026-10-12T06:52:30.165170+00:00 | source (read-only): /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1 | output: /content/drive/MyDrive/research datasets/TGNN_Tardis_TradeDiag_BTCUSDT_2024-05-01_v1 | network used: NO | downloads: NONE
Scope: reference = FULL-DEPTH (unpruned) replay. Categories are mutually exclusive. Previous audit thresholds/results untouched. Not a validation of anything: it describes how trades relate to the rebuilt book.

[A] INPUTS
  binance_incremental_book_L2_2024-05-01_BTCUSDT